In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.append(str(Path("..").resolve()))

from src.data_prepare import prepare_base_df, add_per90

DATA_DIR = Path("..") / "data"
MIN_MINUTES = 900

df_base = prepare_base_df(DATA_DIR, min_minutes=MIN_MINUTES, drop_gk=True)

df_base.shape, df_base.columns[:]

((2511, 123),
 Index(['2CrdY', '90s', 'AerLost', 'AerWon', 'AerWon%', 'Age', 'Assists',
        'BlkPass', 'BlkSh', 'Blocks',
        ...
        'TouAtt3rd', 'TouAttPen', 'TouDef3rd', 'TouDefPen', 'TouLive',
        'TouMid3rd', 'Touches', 'Pos_tags', 'Is_hybrid', 'Pos_group'],
       dtype='str', length=123))

In [6]:
FEATURE_GROUPS = {
    # 1) Strzelanie / threat
    "threat": {
        "weight": 0.18,
        "per90": ["Goals", "Shots", "SoT"],
        "raw": ["SoT%", "G/Sh", "ShoDist"],
    },

    # 2) Kreacja
    "creation": {
        "weight": 0.15,
        "per90": ["Assists", "PasAss", "SCA", "GCA", "PPA", "CrsPA"],
        "raw": [],
    },

    # 3A) Progresja podaniowa
    "progression_pass": {
        "weight": 0.10,
        "per90": ["PasProg", "Pas3rd", "TB", "Sw", "PasTotPrgDist"],
        "raw": [],
    },

    # 3B) Progresja carry / ball movement
    "progression_carry": {
        "weight": 0.08,
        "per90": ["CarProg", "CarPrgDist", "CPA", "RecProg"],
        "raw": [],
    },

    # 4A) Wolumen podań (ile + jak daleko)
    "passing_volume": {
        "weight": 0.09,
        "per90": ["PasTotAtt", "PasTotCmp", "PasTotDist"],
        "raw": [],
    },

    # 4B) Jakość podań (celność)
    "passing_quality": {
        "weight": 0.05,
        "per90": [],
        "raw": ["PasTotCmp%"],
    },

    # 5) Touch profile (rola)
    "touch_profile": {
        "weight": 0.10,
        "per90": ["Touches", "TouDef3rd", "TouMid3rd", "TouAtt3rd", "TouAttPen"],
        "raw": [],
    },

    # 6) Defensywa (lekko w górę)
    "defense_actions": {
        "weight": 0.15,
        "per90": ["Tkl", "Int", "Tkl+Int", "Blocks", "Clr", "Err"],
        "raw": [],
    },

    # 7) Aerial / fizyczność
    "aerial": {
        "weight": 0.06,
        "per90": ["AerWon", "AerLost"],
        "raw": ["AerWon%"],
    },

    # 8) Ball security
    "ball_security": {
        "weight": 0.04,
        "per90": ["CarMis", "CarDis"],
        "raw": [],
    },
}

In [7]:
per90_cols = sorted({c for g in FEATURE_GROUPS.values() for c in g["per90"]})
raw_cols  = sorted({c for g in FEATURE_GROUPS.values() for c in g["raw"]})

missing = [c for c in (per90_cols + raw_cols) if c not in df_base.columns]
print("Missing features:", missing)

per90_cols = [c for c in per90_cols if c in df_base.columns]
raw_cols = [c for c in raw_cols if c in df_base.columns]

print("Using per90:", len(per90_cols), "Using raw:", len(raw_cols))

Missing features: []
Using per90: 36 Using raw: 5


In [8]:
for c in raw_cols:
    df_base[c] = pd.to_numeric(df_base[c], errors="coerce")

df_base = add_per90(df_base, per90_cols)

feature_cols_final = [c + "_per90" for c in per90_cols] + raw_cols
X = df_base[feature_cols_final].copy()

print("X shape:", X.shape)
X.isna().mean().sort_values(ascending=False).head(10)

X shape: (2511, 41)


AerLost_per90       0.0
AerWon_per90        0.0
Assists_per90       0.0
Blocks_per90        0.0
CPA_per90           0.0
CarDis_per90        0.0
CarMis_per90        0.0
CarPrgDist_per90    0.0
CarProg_per90       0.0
Clr_per90           0.0
dtype: float64

In [9]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

imputer = SimpleImputer(strategy="median")
scaler = RobustScaler()

X_imp = imputer.fit_transform(X)
X_scaled = scaler.fit_transform(X_imp)

X_scaled.shape

(2511, 41)

In [10]:
import numpy as np

X_weighted = X_scaled.copy()
col_index = {col: i for i, col in enumerate(feature_cols_final)}

for group_name, cfg in FEATURE_GROUPS.items():
    w = cfg["weight"]
    cols = [c + "_per90" for c in cfg["per90"]] + cfg["raw"]
    cols = [c for c in cols if c in feature_cols_final]
    factor = np.sqrt(w)
    for c in cols:
        X_weighted[:, col_index[c]] *= factor

X_weighted.shape

(2511, 41)

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(X_weighted)
sim.shape

(2511, 2511)

In [12]:
def find_similar(player_name: str, season: str, top_n: int = 10, power: int = 3):
    mask = (df_base["Player"] == player_name) & (df_base["Season"] == season)
    if mask.sum() == 0:
        raise ValueError("Nie znaleziono takiego Player + Season.")

    idx = df_base.index[mask][0]
    idx_pos = df_base.index.get_loc(idx)

    scores = sim[idx_pos].copy()

    # nie pokazuj siebie
    scores[idx_pos] = -1

    # nie pokazuj innych sezonów tego samego zawodnika
    same_player = (df_base["Player"].values == player_name)
    scores[same_player] = -1

    # kalibracja (ładne similarity)
    scores_cal = scores.copy()
    pos = scores_cal > 0
    scores_cal[pos] = scores_cal[pos] ** power

    top_idx = np.argsort(scores_cal)[::-1][:top_n]

    result = df_base.iloc[top_idx][
        ["Player", "Season", "Squad", "Comp", "Pos", "Min"]
    ].copy()

    result["similarity"] = scores_cal[top_idx].round(3)

    return result.sort_values("similarity", ascending=False)

In [13]:
df_base[["Player","Season"]].tail(100)

,Player,Season
2411,Martin Terrier,2022/23
2412,Tetê,2022/23
2413,Kenny Tete,2022/23
2414,Arthur Theate,2022/23
2415,Adrien Thomasson,2022/23
...,...,...
2506,Kurt Zouma,2022/23
2507,Igor Zubeldia,2022/23
2508,Martín Zubimendi,2022/23
2509,Martin Ødegaard,2022/23


In [14]:
find_similar("Yannick Carrasco", "2022/23", power=4)

,Player,Season,Squad,Comp,Pos,Min,similarity
1703,Samuel Chukwueze,2022/23,Villarreal,La Liga,FWMF,1008,0.605
653,Jonathan Ikone,2021/22,Lille,Ligue 1,MFFW,1062,0.581
296,Joaquín Correa,2021/22,Inter,Serie A,FW,1025,0.571
2259,Nemanja Radonji?,2022/23,Torino,Serie A,MFFW,916,0.553
373,Luis Díaz,2021/22,Liverpool,Premier League,FW,958,0.526
1281,Stephan El Shaarawy,2021/22,Roma,Serie A,DFFW,1055,0.524
2266,Raphinha,2022/23,Barcelona,La Liga,FW,959,0.458
861,Riyad Mahrez,2021/22,Manchester City,Premier League,FW,1498,0.453
1470,Nico Williams,2021/22,Athletic Club,La Liga,MF,1337,0.442
287,Kingsley Coman,2021/22,Bayern Munich,Bundesliga,FWDF,1359,0.441


In [15]:
def compare_players(base_player: str, base_season: str, top_n: int = 5):
    mask = (df_base["Player"] == base_player) & (df_base["Season"] == base_season)
    if mask.sum() == 0:
        raise ValueError("Nie znaleziono takiego Player + Season.")

    base_idx = df_base.index[mask][0]
    base_pos = df_base.index.get_loc(base_idx)

    scores = sim[base_pos].copy()

    # wyklucz siebie
    scores[base_pos] = -1

    # wyklucz inne sezony tego samego zawodnika
    same_player = (df_base["Player"].values == base_player)
    scores[same_player] = -1

    top_idx_pos = np.argsort(scores)[::-1][:top_n]
    top_indices = df_base.index[top_idx_pos]


    selected_rows = df_base.loc[[base_idx] + list(top_indices)]
    stats_table = selected_rows[["Player", "Season"] + feature_cols_final].copy()

    return stats_table.round(3)

In [16]:
compare_players("Lionel Messi", "2022/23", top_n=5)

,Player,Season,AerLost_per90,AerWon_per90,Assists_per90,Blocks_per90,CPA_per90,CarDis_per90,CarMis_per90,CarPrgDist_per90,...,TouAtt3rd_per90,TouAttPen_per90,TouDef3rd_per90,TouMid3rd_per90,Touches_per90,AerWon%,G/Sh,PasTotCmp%,ShoDist,SoT%
2135,Lionel Messi,2022/23,0.010,0.000,0.033,0.023,0.072,0.141,0.082,10.086,...,2.189,0.339,0.160,2.171,4.474,0.0,0.13,80.1,17.8,42.5
1575,Alex Baena,2022/23,0.076,0.000,0.008,0.068,0.061,0.137,0.137,7.174,...,1.835,0.287,0.530,1.661,3.965,0.0,0.12,77.5,23.4,38.2
2331,Muhammed Saracevic Cham,2022/23,0.016,0.016,0.011,0.054,0.032,0.216,0.135,8.397,...,1.713,0.195,0.514,2.029,4.199,50.0,0.10,78.4,20.1,34.5
1768,Brahim Díaz,2022/23,0.083,0.015,0.023,0.068,0.068,0.219,0.295,10.409,...,1.591,0.250,0.272,2.287,4.043,15.4,0.19,78.6,16.2,38.1
1819,Bruno Fernandes,2022/23,0.072,0.023,0.009,0.048,0.016,0.068,0.052,3.771,...,1.324,0.166,0.470,1.171,2.943,23.8,0.09,74.1,21.6,30.2
2509,Martin Ødegaard,2022/23,0.037,0.026,0.014,0.026,0.051,0.080,0.066,4.487,...,1.508,0.217,0.300,1.267,3.037,40.9,0.16,79.1,18.9,31.4
